In [ ]:
#@title Import all necessary libraries

# General python library
import os
import cv2
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
import matplotlib.animation as animation

# Torch library
import torch
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch import nn
from torchvision import datasets
from torchvision.transforms import ToTensor

# Colab
from google.colab import drive

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

In [ ]:
#@title Useful functions

def load_image(filename):
  im_pil = Image.open(os.path.join(srcpath, filename))
  im = np.array(im_pil).astype(np.float32) / 255
  return im

def train(dataloader,
          model,
          loss_fn,
          optimizer,
          loss_print_iter: int=100
          ) -> None:
    size = len(dataloader.dataset)

    # Set the model to the training mod
    model.train()

    for batch, (image, label) in enumerate(dataloader):
        image, label = image.to(device), label.to(device)

        # Compute prediction error
        pred = model(image)
        loss = loss_fn(pred, label)

        # Backpropagation

        # Computes gradients of the loss with respect to all parameters
        # in the model that require gradients, storing them in the .grad
        # attribute of each parameter.
        loss.backward()

        # Updates all the model parameters using the gradients computed in the
        # previous step, moving parameters in the direction that reduces the
        # loss.
        optimizer.step()

        # Clear gradient for next iterations.
        optimizer.zero_grad()

        if batch % 100 == 0:
          loss, current = loss.item(), (batch + 1) * len(image)
          print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
#@title At last, mount your colab

drive.mount('/content/drive')
srcpath = 'your_srcpath'  #@param {type:'string'}
srcpath = os.path.join('/content/drive/My Drive', srcpath)
print('srcpath = ', srcpath)

In [ ]:
#@title Load the MNIST dataset

training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

# Create data loaders.
batch_size = 64    #@param {type:"integer"}
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [ ]:
#@title Print shapes of the input image and the output label

# Note that this channel order in torch is slightly different from slides.

for input_image, _ in test_dataloader:
    print(f"Shape of input image: [Number of images in a batch, No. of channel, Height, Width]: {input_image.shape}")
    break

In [ ]:
#@title Visualize the training 10 samples

plt.figure(figsize=(40, 10))
for input_image, ground_truth_label in test_dataloader:
  for i in range(10):
      plt.subplot(2, 5, i + 1)
      plt.imshow(input_image[i, 0, ...], cmap='gray')
      label_i = int(ground_truth_label[i])
      plt.title(f'Number {label_i}', fontsize=18)
  break

In [ ]:
#@title Define **loss**

loss_fn = nn.CrossEntropyLoss()

In [ ]:
#@title Also setup a training device.

device = 'cpu'  # For this tutorial, cpu is enough. For more efficient training, you could GPU ('cuda'). However, you need to purchase GPU hours if you want to do so.

In [ ]:
#@title 3.1. Define the network

class NeuralNetwork1(nn.Module):
    def __init__(self):
        super().__init__()
        #################################################################
        ### This block is very important. Make sure you understand it ###
        self.network = nn.Sequential(
            nn.Flatten(),             # This flatten image, which converts 28x28 image to 756 vector
            nn.Linear(28*28, 128),    # nn.Linear(a, b) means the input is a channels and the output is b chnnals
            nn.Linear(128, 128),      # Similarly, this is 128-channel input and 128-channel output.
            nn.Linear(128, 10)        # Since the output is 10-class, so the output must be 10 channels.
        )
        #################################################################

    def forward(self, x):
        return self.network(x)

In [ ]:
model_1 = NeuralNetwork1().to(device)
learning_rate = 1e-1
optimizer = torch.optim.SGD(model_1.parameters(), lr=learning_rate)

In [ ]:
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_1, loss_fn, optimizer)
    test(test_dataloader, model_1, loss_fn)
print("Done!")

In [ ]:
#@title Define network

class NeuralNetwork3(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                                                 # A convolutional network
                                                 # After this layer, the image is 28x28 with 32 channels
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                  # nn.MaxPool2d(2, 2) is a 2x2 pooling layer
                                                 # After this layer, the image is 14x14 with 32 channels
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
                                                 # After this layer, the image is still 14x14 with 32 channels
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                  # After this layer, the image is ?x? with 32 channels
            nn.Flatten(),
        #################################################################
        # This line has a bug, please fix it.
            nn.Linear(7*7*32, 10)   # Here we assume that input is 14x14 image with 32 channel, but this is actually wrong. Please fix it.
        #################################################################
        )

    def forward(self, x):
        return self.network(x)

# Individual Project Task 1
Train CNN for 10 Epochs and Plot Accuracy Curve

In [ ]:
#import library
import time
import matplotlib.pyplot as plt

In [ ]:
def test_and_return_acc(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    return 100 * (correct / size)

In [ ]:
#10 Epochs Training
model_cnn = NeuralNetwork3().to(device)
learning_rate = 1e-1
optimizer = torch.optim.SGD(model_cnn.parameters(), lr=learning_rate)
epochs = 10
history_acc = []

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_cnn, loss_fn, optimizer)
    test(test_dataloader, model_cnn, loss_fn)
    acc = test_and_return_acc(test_dataloader, model_cnn, loss_fn)
    history_acc.append(acc)
print("Done!")


plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), history_acc, marker='o', color='b')
plt.title("CNN Training: Epoch vs Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.show()

Batch size comparison

In [ ]:
batch_sizes = [16, 64, 256, 1024]
bs_accuracy = []
bs_runtime = []

for bs in batch_sizes:
    print(f"Batch Size {bs}\n-------------------------------")
    # Re-prepare data loaders
    tmp_train_loader = DataLoader(training_data, batch_size=bs, shuffle=True)
    tmp_test_loader = DataLoader(test_data, batch_size=bs)

    model_tmp = NeuralNetwork3().to(device)
    opt_tmp = torch.optim.SGD(model_tmp.parameters(), lr=1e-1)

    start_time = time.time()
    # Train for 1 epoch to measure runtime
    train(tmp_train_loader, model_tmp, loss_fn, opt_tmp)
    end_time = time.time()

    acc = test_and_return_acc(tmp_test_loader, model_tmp, loss_fn)

    bs_runtime.append(end_time - start_time)
    bs_accuracy.append(acc)
    print(f"Acc {acc:.2f}%, Time {end_time - start_time:.2f}s")

# Plotting Batch Size Results
# --- Plot 1: Batch Size vs Accuracy ---
plt.figure(figsize=(8, 5))
plt.plot(batch_sizes, bs_accuracy, color='tab:blue', marker='s', linewidth=2)
plt.title("Impact of Batch Size on Model Accuracy")
plt.xlabel("Batch Size")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# --- Plot 2: Batch Size vs Runtime ---
plt.figure(figsize=(8, 5))
plt.plot(batch_sizes, bs_runtime, color='tab:red', marker='o', linestyle='--', linewidth=2)
plt.title("Impact of Batch Size on Training Runtime")
plt.xlabel("Batch Size")
plt.ylabel("Runtime (seconds)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

Batch Size 16
-------------------------------


NameError: name 'training_data' is not defined

Learning Rate Exploration

In [ ]:
learning_rates = [1e-3, 1e-2, 1e-1, 1.0]
lr_results = []

for lr in learning_rates:
    model_lr = NeuralNetwork3().to(device)
    optimizer = torch.optim.SGD(model_lr.parameters(), lr=lr)
    print(f"LR: {lr}\n-------------------------------")
    # Train for 3 epochs
    for t in range(3):
        train(train_dataloader, model_lr, loss_fn, optimizer)
        test(test_dataloader, model_lr, loss_fn)
    acc = test_and_return_acc(test_dataloader, model_lr, loss_fn)
    lr_results.append(acc)
    print(f"Accuracy {acc:.2f}%\n")

plt.figure(figsize=(8, 5))
plt.semilogx(learning_rates, lr_results, marker='D', color='green')
plt.title("Learning Rate & Accuracy (after 3 epochs)")
plt.xlabel("Learning Rate")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.show()

Implement a new network by changing ReLu to Sigmoid

**The code is directly copy from NeuralNetwork3**


In [ ]:
class NeuralNetworkSigmoid(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                                                 # A convolutional network
                                                 # After this layer, the image is 28x28 with 32 channels
            nn.Sigmoid(),
            nn.MaxPool2d(2, 2),                  # nn.MaxPool2d(2, 2) is a 2x2 pooling layer
                                                 # After this layer, the image is 14x14 with 32 channels
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
                                                 # After this layer, the image is still 14x14 with 32 channels
            nn.Sigmoid(),
            nn.MaxPool2d(2, 2),                  # After this layer, the image is ?x? with 32 channels
            nn.Flatten(),
        #################################################################
        # This line has a bug, please fix it.
            nn.Linear(7*7*32, 10)   # Here we assume that input is 14x14 image with 32 channel, but this is actually wrong. Please fix it.
        #################################################################
        )

    def forward(self, x):
        return self.network(x)

Test the training result and compare it with the neural network used in Task 1, for 10 epochs

In [ ]:
model_sigmoid = NeuralNetworkSigmoid().to(device)
learning_rate = 1e-1
optimizer = torch.optim.SGD(model_sigmoid.parameters(), lr=learning_rate)
epochs = 10
history_accS = []
print(f"Neurak Network (using Sigmoid)\n")
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model_sigmoid, loss_fn, optimizer)
    test(test_dataloader, model_sigmoid, loss_fn)
    acc = test_and_return_acc(test_dataloader, model_sigmoid, loss_fn)
    history_accS.append(acc)
print("Done!")

plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), history_accS, color='tab:blue', marker='s', linewidth=2)
plt.title("New Neural Network (Sigmoid)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), history_acc, color='tab:red', marker='o', linestyle='--', linewidth=2)
plt.title("Original Neural Network")
plt.xlabel("Epoch")
plt.ylabel("Runtime (seconds)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

#Individual Project Task 2
# Denoising image

*Create a dataset*

In [ ]:
training_data_denoising = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

testing_data_denoising = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

clean_train = training_data_denoising.data.numpy()
clean_train = clean_train.astype(np.float32) / 255.0

clean_test = testing_data_denoising.data.numpy()
clean_test = clean_test.astype(np.float32) / 255.0

Adding noise

In [ ]:
sigma = 0.3

# Create noisy training data
noisy_train = clean_train + sigma* np.random.normal(0, 1, size= clean_train.shape)

# Create noisy test data
noisy_test = clean_test + sigma* np.random.normal(0, 1, size= clean_test.shape)

# Cliping
#noisy_train = np.clip(noisy_train, 0., 1.)
#noisy_test = np.clip(noisy_test, 0., 1.)

In [ ]:
# Prepare Training Dataset
train_ds = TensorDataset(torch.from_numpy(noisy_train).unsqueeze(1),torch.from_numpy(clean_train).unsqueeze(1))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# Prepare Test Dataset
test_ds = TensorDataset(torch.from_numpy(noisy_test).unsqueeze(1),torch.from_numpy(clean_test).unsqueeze(1))
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

Comparison between noisy and clean image

In [ ]:
# show original image
index = np.random.randint(len(clean_test))
plt.imshow(clean_test[index].reshape(28,28))
plt.show()
# show noisy image
plt.imshow(noisy_test[index].reshape(28,28))
plt.show()

Implement a simplified U-Net architecture

In [ ]:
class UNetDenoiser(nn.Module):
    def __init__(self):
        super(UNetDenoiser, self).__init__()

        # Encoder: Downsampling
        self.enc1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) # Reduces 28x28 -> 14x14

        # Decoder: Upsampling
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1 = nn.Conv2d(32 + 16, 16, kernel_size=3, padding=1) # +16 for skip connection
        self.dec2 = nn.Conv2d(16, 1, kernel_size=3, padding=1)

        self.relu = nn.ReLU() # Standard non-linear operator

    def forward(self, x):
        # Encoder
        x1 = self.relu(self.enc1(x))
        x2 = self.pool(x1)
        x2 = self.relu(self.enc2(x2))

        # Decoder with Skip Connection
        x_up = self.upsample(x2)
        # Concatenate original encoder features (x1) with upsampled features
        x_cat = torch.cat([x_up, x1], dim=1)

        x_out = self.relu(self.dec1(x_cat))
        x_out = torch.sigmoid(self.dec2(x_out)) # Final layer to keep output in [0, 1] range
        return x_out

**Setup MSE Loss**

In [ ]:
criterion = nn.MSELoss()

*Add the training process*

**By feeding the model with nousy images and calculating the loss based on the original image**

In [ ]:
import torch.optim as optim

# Initialization
model = UNetDenoiser().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for noisy_imgs, clean_imgs in train_loader:
        noisy_imgs, clean_imgs = noisy_imgs.to(device).float(), clean_imgs.to(device).float()

        # Forward pass
        outputs = model(noisy_imgs)
        loss = criterion(outputs, clean_imgs)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}")

Running the denoise model on noisy input

In [ ]:
model.eval()
with torch.no_grad():
    # Get a batch of test images
    noisy_test, clean_test = next(iter(test_loader))
    noisy_test = noisy_test.to(device).float()

    # Get denoised output
    denoised_output = model(noisy_test).cpu()

Noisy input

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(3):
    # Plot Noisy Input
    plt.subplot(2, 3, i + 1)
    plt.imshow(noisy_test[i].cpu().squeeze(), cmap='gray')
    plt.title("Noisy input")
    plt.axis('off')

# See the result

Denoise output

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(3):
    # Plot Noisy Input
    plt.subplot(2, 3, i + 1)
    plt.imshow(denoised_output[i].cpu().squeeze(), cmap='gray')
    plt.title("Denoised output")
    plt.axis('off')